In [0]:
# Setup da camada Gold
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

DataFrame[]

# **gold.dim_movies**

In [0]:
# Gera-se as chaves substitutas (sk_movie_id, sk_person_id, sk_company_id) usando row_number/monotonically_increasing_id
# Garante-se que cada dimensão contenha entidades únicas e deduplicadas

# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.dim_movies")

# Garante unicidade antes da SK para não atribuir chaves diferentes ao mesmo filme
df_filmes_base = (spark.table("silver.tb_info_filmes")
    .dropDuplicates(["id_filme"])
)

# Cria a dimensão com a Surrogate Key (sk_movie_id)
df_dim_movies = df_filmes_base.select(
    F.monotonically_increasing_id().cast("BIGINT").alias("sk_movie_id"),
    F.col("id_filme").cast("STRING").alias("id_filme"),
    F.col("titulo").cast("STRING"),
    F.col("data_lancamento").cast("DATE"),
    F.col("ano_lancamento").cast("INT"),
    F.col("duracao_minutos").cast("INT"),
    F.col("idioma_original").cast("STRING"),
    F.col("status_filme").cast("STRING"),
    F.col("sinopse").cast("STRING")
)

# Salva a dimensão em Delta
df_dim_movies.write.format("delta").mode("overwrite").saveAsTable("gold.dim_movies")

display(df_dim_movies.limit(10))


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
0,1000014,On va manquer !,2018-05-15,2018,0,fr,Lançado,null
1,1000079,Pourquoi tu souris ?,2024-06-26,2024,0,fr,Em Produção,null
2,1000409,Strangers by Night,2023-07-05,2023,90,fr,Lançado,\In a crowded subway train
3,1000521,Princes of the Desert,2023-02-08,2023,105,fr,Lançado,"Zodi, a 12 years old nomad, find an orphan baby dromedary in the desert. He takes in the animal, feeds it, calls it Téhu and becomes its best friend."
4,1000948,A Monstrous Marriage,2023-06-21,2023,101,it,Lançado,"Vladimiro and Brunilde are grappling with the disappearance of Nando, the head of the family. The sad event brought together the human family of Luna, daughter of Nando, and the monstrous one of her husband Adalberto. Humans, vampires, witches, werewolves and ghosts will find themselves in the throes of a new monstrous marriage..."
5,1001598,Go Chase Yourself,2022-07-18,2022,0,en,Lançado,null
6,1001636,In The Month of July,2021-03-05,2021,81,hi,Lançado,"The film has been approached with a light and breezy treatment which includes eye-pleasing visuals as it is primarily shot on Red Epic format, in scenic locales of Himachal Pradesh, melodious music, appealing faces and subtle performances."
7,1001946,Past Lives,2022-07-16,2022,1,en,Lançado,Two men who are not meant to be together experience a connection so strong and meaningful as if they knew each other from previous lives.
8,1001974,Flood in the Desert,2022-05-03,2022,53,en,Lançado,null
9,1004908,The Great Awakening,2022-08-05,2022,60,en,Lançado,null


# **gold.dim_genres**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.dim_genres")

# Cria um catálogo único de gêneros
df_dim_genres = (spark.table("silver.tb_generos")
    .select("nome_genero")
    # Filtro para evitar que nulos ou strings vazias entrem no catálogo
    .filter(F.col("nome_genero").isNotNull() & (F.trim(F.col("nome_genero")) != ""))
    .distinct()
    .select(
        F.monotonically_increasing_id().cast("BIGINT").alias("sk_genre_id"),
        F.col("nome_genero").cast("STRING")
    )
)

# Salva a dimensão em Delta
df_dim_genres.write.format("delta").mode("overwrite").saveAsTable("gold.dim_genres")

display(df_dim_genres)

sk_genre_id,nome_genero
0,Science Fiction
1,Adventure
2,Crime
3,Comedy
4,Drama
5,Fantasy
6,Action
7,Thriller
8,Mystery
9,Family


# **gold.dim_people**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.dim_people")

# Filtra pessoas físicas e deduplica primeiro
df_dim_people = (spark.table("silver.tb_pessoas_empresas")
    # Filtra apenas pessoas físicas             
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.trim(F.col("nome_entidade")).alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    # Descarta nulos ou vazios para manter a qualidade da dimensão
    .filter(F.col("nome_pessoa").isNotNull() & (F.col("nome_pessoa") != ""))
    .distinct()
    
    # Gera o ID único para cada pessoa/papel
    .select(
        F.monotonically_increasing_id().cast("BIGINT").alias("sk_person_id"),
        F.col("nome_pessoa").cast("STRING"),
        F.col("tipo_pessoa").cast("STRING")
    )
)

# Salva a dimensão em Delta
df_dim_people.write.format("delta").mode("overwrite").saveAsTable("gold.dim_people")

display(df_dim_people.limit(10))

sk_person_id,nome_pessoa,tipo_pessoa
0,Gary Weeks,Ator
1,Bobby Schofield,Ator
2,Kaitlyn Dever,Ator
3,Eddie Redmayne,Ator
4,Damien Molony,Ator
5,Bella Thorne,Ator
6,Toby Jones,Ator
7,Manuela Ventura,Ator
8,Giampaolo Morelli,Ator
9,Christina Crevillén,Ator


# **gold.dim_companies**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.dim_companies")

# Filtra produtoras e pega apenas os nomes distintos
df_dim_companies = (spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.trim(F.col("nome_entidade")).alias("nome_produtora"))
    # Filtro para não catalogar produtoras sem nome ou vazias
    .filter(F.col("nome_produtora").isNotNull() & (F.col("nome_produtora") != ""))
    .distinct()  
    
    .select(
        F.monotonically_increasing_id().cast("BIGINT").alias("sk_company_id"),
        F.col("nome_produtora").cast("STRING")
    )
)

# Salva a dimensão em Delta
df_dim_companies.write.format("delta").mode("overwrite").saveAsTable("gold.dim_companies")

display(df_dim_companies.limit(10))

sk_company_id,nome_produtora
0,Universal Pictures
1,Toho Music
2,Cinesite
3,La Claqueta
4,Aurora Alliance
5,The New York Times
6,Brad Krevoy Television
7,Goalpost Pictures
8,Studio O
9,Cien Cine


# **gold.dim_reviews**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.dim_reviews")

df_movies = spark.table("gold.dim_movies").select("sk_movie_id", "id_filme")

# Resume as avaliações dos usuários por filme
df_dim_reviews = (spark.table("silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("INT").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("DOUBLE").alias("nota_media_usuarios")
    )
    .join(df_movies, on="id_filme", how="inner")
    
    # Gera a Surrogate Key e mantém as colunas do PDF
    .withColumn("sk_review_id", F.monotonically_increasing_id().cast("BIGINT"))
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

# Salva a dimensão em Delta
df_dim_reviews.write.format("delta").mode("overwrite").saveAsTable("gold.dim_reviews")

display(df_dim_reviews.limit(10))

sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
0,0,1,3.3
1,9,1,6.1
2,12,1,0.6
3,15,1,1.9
4,16,1,0.6
5,23,1,1.0
6,27,1,2.9
7,31,1,3.3
8,32,1,4.2
9,34,1,0.3


# **gold.bridge_movie_genre**

In [0]:
# Tabelas-ponte (Bridge Tables):
# Um filme pode ter vários gêneros, vários atores e várias produtoras (relação N para N)
# Usa-se as bridge tables para ligar os filmes a essas dimensões sem duplicar as linhas da tabela fact


# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.bridge_movie_genre")

# Relaciona cada filme aos seus gêneros
df_bridge_genre = (spark.table("silver.tb_generos")
    .join(
        spark.table("gold.dim_movies").select("sk_movie_id", "id_filme"),
        "id_filme",
        "inner"
    )
    .join(spark.table("gold.dim_genres"),
        "nome_genero",
        "inner"
    )
    .select(
        "sk_movie_id",
        "sk_genre_id"
    )
    .dropDuplicates()
)

# Salva a tabela ponte em Delta
df_bridge_genre.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_genre")

display(df_bridge_genre.limit(10))


sk_movie_id,sk_genre_id
1,16
26,0
35,2
44,0
49,0
52,6
54,14
128,6
132,16
136,16


# **gold.bridge_movie_person**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.bridge_movie_person")

# Relaciona cada filme às pessoas envolvidas
df_bridge_person = (spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(
        spark.table("gold.dim_movies").select("sk_movie_id", "id_filme"),
        "id_filme",
        "inner"
    )
    # Join com a dim_people para buscar a sk_person_id
    .join(spark.table("gold.dim_people"),
        (F.col("nome_entidade") == F.col("nome_pessoa")) &
        (F.col("tipo_entidade") == F.col("tipo_pessoa")),
        "inner"
    )
    .select(
        "sk_movie_id",
        "sk_person_id"
    )
    .dropDuplicates()
)

# Salva a tabela ponte em Delta
df_bridge_person.write.format("delta").mode("overwrite").saveAsTable("gold.bridge_movie_person")

display(df_bridge_person.limit(10))


sk_movie_id,sk_person_id
34509,242417
95667,97320
69094,242481
79406,24463
41910,73093
93573,263
58975,317
46241,218321
19503,73258
64555,379


# **gold.bridge_movie_company**

In [0]:
# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.bridge_movie_company")

# Relaciona cada filme às suas produtoras
df_bridge_company = (spark.table("silver.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(
        spark.table("gold.dim_movies").select("sk_movie_id", "id_filme"),
        "id_filme",
        "inner"
    )
    # Join com a dim_companies para buscar a sk_company_id
    .join(spark.table("gold.dim_companies"),
        F.col("nome_entidade") == F.col("nome_produtora"),
        "inner"
    )
    .select(
        "sk_movie_id",
        "sk_company_id"
    )
    .dropDuplicates()
)

# Salva a tabela ponte em Delta
df_bridge_company.write.format("delta").mode("overwrite").saveAsTable(
    "gold.bridge_movie_company"
)

display(df_bridge_company.limit(10))


sk_movie_id,sk_company_id
95,16858
109,2337
168,33530
206,33869
281,22562
298,9327
412,4883
555,27369
600,16450
658,27060


# **gold.fact_movies_performance**

In [0]:
# Um registro único por filme
# Consolida as métricas financeiras (orçamento, receita, lucro) e de engajamento (popularidade, votos, notas)
# Filtrada apenas para filmes lançados, garantindo que não se duplique ao juntar com as dimensões


# Remove a tabela anterior, caso exista
spark.sql("DROP TABLE IF EXISTS gold.fact_movies_performance")

# Pega os filmes lançados direto da dim_movies
df_filmes_lancados = (
    spark.table("gold.dim_movies")
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
)

# Joins passando apenas "id_filme"
df_fact = (
    df_filmes_lancados
    .join(spark.table("silver.tb_financeiro_filmes"), "id_filme", "left")
    .join(spark.table("silver.tb_metricas_engajamento"), "id_filme", "left")
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("DECIMAL(18,2)"),
        F.col("receita_usd").cast("DECIMAL(18,2)"),
        F.col("lucro_usd").cast("DECIMAL(18,2)"),
        F.col("orcamento_brl").cast("DECIMAL(18,2)"),
        F.col("receita_brl").cast("DECIMAL(18,2)"),
        F.col("lucro_brl").cast("DECIMAL(18,2)"),
        "popularidade",
        "nota_media_tmdb",
        F.col("qtd_votos_tmdb").cast("INT"),
        "nota_media_imdb",
        F.col("qtd_votos_imdb").cast("INT")
    ).dropDuplicates(["sk_movie_id"])
)

# Salva a tabela fato em Delta
df_fact.write.format("delta").mode("overwrite").saveAsTable("gold.fact_movies_performance")

display(df_fact.limit(10))

sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
72853,58000000.00,null,null,289588200.00,null,null,72.735,7.606,28894,8.0,1270339
786,40000000.00,171539887.00,131539887.00,199716000.00,856481501.80,656765501.80,88.285,7.323,12007,7.4,578161
92679,100000000.00,null,null,499290000.00,null,null,44.356,null,11395,8.2,784874
92296,200000000.00,654855901.00,454855901.00,998580000.00,3269630028.10,2271050028.10,29.783,6.876,10203,6.5,334724
38129,200.00,null,null,998.58,null,null,56.256,7.303,9298,6.6,488373
61509,25000000.00,235956898.00,210956898.00,124822500.00,1178109196.02,1053286696.02,70.299,8.058,8958,7.8,null
89808,200000000.00,49750471.00,-150249529.00,998580000.00,248399126.66,-750180873.34,60.762,7.865,7625,null,227594
82621,8500000.00,87547518.00,79047518.00,42439650.00,437116002.62,394676352.62,49.553,7.8,null,7.6,314257
25467,37000000.00,117501013.00,80501013.00,184737300.00,586670807.81,401933507.81,18.92,6.85,5372,6.9,283820
69314,175000000.00,148675066.00,-26324934.00,873757500.00,742319737.03,-131437762.97,33.349,null,5333,null,249609
